In [ ]:
class LoudManager:
    def __init__(self, data):
        print(f'LoudManager.__init__(), with {data=}')
        self.data = data
    
    def __enter__(self):
        print(f'LoudManager.__enter__(), with {vars(self)}')
        return self.data
    
    def __exit__(self, exc_type, exc_value, traceback):
        print(f'LoudManager.__exit__(), with {exc_type=}, {exc_value=}, {traceback=}')
        return False  # propagate exception if any
    
lm = LoudManager('my data')

with lm as data:
    print(f'Inside with block, {data=}')

    raise ValueError('Something went wrong!')

LoudManager.__init__(), with data='my data'
LoudManager.__enter__(), with {'data': 'my data'}
Inside with block, data='my data'
LoudManager.__exit__(), with exc_type=<class 'ValueError'>, exc_value=ValueError('Something went wrong!'), traceback=<traceback object at 0x10828b5c0>


ValueError: Something went wrong!

In [ ]:
with open('myfile.txt', mode='w') as f:
    # f.__enter__() is called here
    f.write('Hello, world!')
    # f.__exit__() is called here

!cat myfile.txt

Hello, world!

In [ ]:
import sys

class TempOut(object):
    def __init__(self, filename):
        self.filename = filename
    
    def __enter__(self):
        self.old_stdout = sys.stdout
        self.file = open(self.filename, 'w')
        sys.stdout = self.file
    
    def __exit__(self, exc_type, exc_value, traceback):
        sys.stdout = self.old_stdout
        self.file.close()

In [7]:
class Stats(object):
    def __init__(self, numbers):
        self.numbers = numbers
    
    def min(self):
        return min(self.numbers)
    
    def max(self):
        return max(self.numbers)
    
    def mean(self):
        return sum(self.numbers) / len(self.numbers)
    
    def __enter__(self):
        # filter out any non-numeric values
        self.numbers = [n for n in self.numbers if isinstance(n, (int, float))]
        print(f'Filtered numbers: {self.numbers}')
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        # Invoke all three methods and print the results to screen
        print(f'Min: {self.min()}')
        print(f'Max: {self.max()}')
        print(f'Mean: {self.mean()}')

numbers = [1, 2, 3, 4, 5, 'a', None]

with Stats(numbers) as stats:
    pass

Filtered numbers: [1, 2, 3, 4, 5]
Min: 1
Max: 5
Mean: 3.0


In [3]:
import sqlite3

class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name
        self.connection = None
    
    def __enter__(self):
        self.connection = sqlite3.connect(self.db_name)
        print(f"Connecting to database: {self.db_name}")
        return self.connection.cursor()
    
    def __exit__(self, exc_type, exc_value, traceback):
        if self.connection:
            if exc_type:
                print(f"An error occurred: {exc_value}")
                self.connection.rollback()
            else:
                print("Committing changes to the database.")
                self.connection.commit()
    
        self.connection.close()
        print(f"Connection closed to database: {self.db_name}")
        return False  # propagate exception if any

print("Using DatabaseConnection context manager:")
with DatabaseConnection(':memory:') as cursor:
    cursor.execute('CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT)')
    cursor.execute('INSERT INTO test (name) VALUES (?)', ('Alice',))
    cursor.execute('INSERT INTO test (name) VALUES (?)', ('Bob',))
    cursor.execute('SELECT * FROM test')
    rows = cursor.fetchall()
    print(f"Rows in the database: {rows}")
print()

print("Using DatabaseConnection context manager with an error:")
try:
    with DatabaseConnection(':memory:') as cursor:
        cursor.execute('CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT)')
        cursor.execute('INSERT INTO test (name) VALUES (?)', ('Alice',))
        # This will raise an error because the table already exists
        cursor.execute('CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT)')
except sqlite3.OperationalError as e:
    print(f"Caught an error: {e}")

Using DatabaseConnection context manager:
Connecting to database: :memory:
Rows in the database: [(1, 'Alice'), (2, 'Bob')]
Committing changes to the database.
Connection closed to database: :memory:

Using DatabaseConnection context manager with an error:
Connecting to database: :memory:
An error occurred: table test already exists
Connection closed to database: :memory:
Caught an error: table test already exists


In [5]:
import time
from collections import defaultdict

class PerformanceTime(object):
    _statistics = defaultdict(list)

    def __init__(self, operation_name):
        self.operation_name = operation_name
        self.start_time = None
        self.end_time = None
    
    def __enter__(self):
        self.start_time = time.perf_counter()
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        self.end_time = time.perf_counter()
        elapsed_time = self.end_time - self.start_time

        if exc_type is None:
            self._statistics[self.operation_name].append(elapsed_time)
            print(f"Operation '{self.operation_name}' took {elapsed_time:.6f} seconds.")
        else:
            print(f"Operation '{self.operation_name}' failed with error: {exc_value}")
    
    @classmethod
    def get_statistics(cls):
        return {op: sum(times) / len(times) for op, times in cls._statistics.items() if times}
    

with PerformanceTime('example_operation'):
    time.sleep(1)  # Simulate a time-consuming operation

with PerformanceTime('example_operation'):
    time.sleep(0.5)  # Simulate a faster operation

try:
    with PerformanceTime('fail'):
        time.sleep(0.2)
        raise ValueError("Simulated failure")
except ValueError as e:
    print(f"Caught an exception: {e}")

print("Performance statistics:")
stats = PerformanceTime.get_statistics()
for operation, avg_time in stats.items():
    print(f"Average time for '{operation}': {avg_time:.6f} seconds")

Operation 'example_operation' took 1.003523 seconds.
Operation 'example_operation' took 0.505032 seconds.
Operation 'fail' failed with error: Simulated failure
Caught an exception: Simulated failure
Performance statistics:
Average time for 'example_operation': 0.754278 seconds


In [6]:
import time
from datetime import datetime, timedelta

class MockAPI(object):
    _calls_made = []
    
    @classmethod
    def make_call(cls, endpoint):
        """ Simulate API call that fails if more than 4 calls made in 10 seconds"""
        cls._calls_made.append(datetime.now())
        recent_calls = [call for call in cls._calls_made if call > datetime.now() - timedelta(seconds=10)]
        if len(recent_calls) > 4:
            raise Exception("429: API rate limit exceeded")
        return {"status": "success", "endpoint": endpoint, "timestamp": datetime.now()}
    

class RateLimitHandler(object):
    def __init__(self, max_retries=3, initial_delay=0.5):
        self.max_retries = max_retries
        self.initial_delay = initial_delay
        self.attempts = 0
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            return False  # No exception, do not suppress
        
        if "429" in str(exc_value):
            if self.attempts < self.max_retries:
                self.attempts += 1
                delay = self.initial_delay * (2 ** (self.attempts - 1))  # Exponential backoff
                print(f"Rate limit hit. Retrying in {delay} seconds... (Attempt {self.attempts}/{self.max_retries})")
                time.sleep(delay)
                return True  # Suppress the exception to retry
            else:
                print("Max retries reached. Giving up.")

        return False  # Propagate other exceptions

MockAPI._calls_made = []  # Reset call history for testing
successful_calls = 0
endpoints = [f"/users/{i}" for i in range(8)]  # 8 endpoints to call

for endpoint in endpoints:
    for attempt in range(4):  # Allow up to 4 attempts per endpoint
        try:
            with RateLimitHandler(max_retries=3, initial_delay=0.5) as handler:
                handler.attempts = attempt  # Set the current attempt number
                response = MockAPI.make_call(endpoint)
                print(f"API call successful: {response}")
                successful_calls += 1
                break  # Exit the retry loop on success
        except Exception as e:
            print(f"API call failed: {e}")
            if "429" not in str(e):
                break  # Exit the retry loop on non-rate-limit errors

API call successful: {'status': 'success', 'endpoint': '/users/0', 'timestamp': datetime.datetime(2026, 7, 2, 21, 2, 15, 20928)}
API call successful: {'status': 'success', 'endpoint': '/users/1', 'timestamp': datetime.datetime(2026, 7, 2, 21, 2, 15, 21043)}
API call successful: {'status': 'success', 'endpoint': '/users/2', 'timestamp': datetime.datetime(2026, 7, 2, 21, 2, 15, 21055)}
API call successful: {'status': 'success', 'endpoint': '/users/3', 'timestamp': datetime.datetime(2026, 7, 2, 21, 2, 15, 21063)}
Rate limit hit. Retrying in 0.5 seconds... (Attempt 1/3)
Rate limit hit. Retrying in 1.0 seconds... (Attempt 2/3)
Rate limit hit. Retrying in 2.0 seconds... (Attempt 3/3)
Max retries reached. Giving up.
API call failed: 429: API rate limit exceeded
Rate limit hit. Retrying in 0.5 seconds... (Attempt 1/3)
Rate limit hit. Retrying in 1.0 seconds... (Attempt 2/3)
Rate limit hit. Retrying in 2.0 seconds... (Attempt 3/3)
Max retries reached. Giving up.
API call failed: 429: API rate l

# Context Manager Protocol
1. We start a block with `with` keyword and an object
2. We can optionally assign it to a variable
3. at the start of the block, object's __enter__ emthod is called
4. at the end of the block objects __exit__ method is called